In [12]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

In [13]:
data = pd.read_csv("ultimate_student_productivity_dataset_5000.csv")
data['pass'] = np.where(data['exam_score'] >= 24, 1, 0)
X = data.select_dtypes(include=np.number)
X = X.drop(columns=['exam_score','student_id','pass'])
y = data['pass']
print(data['pass'].head())

0    1
1    0
2    0
3    0
4    1
Name: pass, dtype: int64


In [14]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)

lr = LogisticRegression(max_iter=5000)
svc = SVC(probability=True)

voting_hard =  VotingClassifier(
    estimators=[('lr',lr),('rf',rf),('svc',svc)],
    voting = 'hard'
)
voting_hard.fit(X_train, y_train)
hard_pred = voting_hard.predict(X_test)

voting_soft =  VotingClassifier(
    estimators=[('lr',lr),('rf',rf),('svc',svc)],
    voting = 'soft'
)
voting_soft.fit(X_train, y_train)
soft_pred = voting_soft.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))
print("XGBoost Accuracy:", accuracy_score(y_test, xgb_pred))
print("Hard Voting Accuracy:", accuracy_score(y_test, hard_pred))
print("Soft Voting Accuracy:", accuracy_score(y_test, soft_pred))
print("\nClassification Report (Soft Voting):\n", classification_report(y_test, soft_pred))

/home/student-35/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [10:06:52] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Random Forest Accuracy: 0.883
XGBoost Accuracy: 0.871
Hard Voting Accuracy: 0.892
Soft Voting Accuracy: 0.891

Classification Report (Soft Voting):
               precision    recall  f1-score   support

           0       0.91      0.93      0.92       667
           1       0.86      0.80      0.83       333

    accuracy                           0.89      1000
   macro avg       0.88      0.87      0.88      1000
weighted avg       0.89      0.89      0.89      1000

